# Run the migration matrix on a Colab T4 (base model, run `base01`)

Translates every benchmark problem along every **priority-1** pair that has a source file, with
**Qwen2.5-Coder-7B-Instruct, 4-bit, no adapter** and prompt `v1`. With Python and Java sources that is
8 pairs x 15 problems = 120 translations; pairs without a source are recorded as `skipped: no_source`.

Outputs go to **Google Drive** so a Colab disconnect loses nothing: the runner skips finished folders, so
just re-run the cell after a disconnect. Afterwards copy the run folder into `results/raw/` in the repo and
score it locally with `python pipeline/collect.py --run-id base01` (needs Docker).

Before running: **Runtime → Change runtime type → T4 GPU.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/CS152/results/raw"
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
RUN_ID = "base01"   # base model, prompt v1, no adapter

In [ ]:
import os, subprocess
REPO = "https://github.com/05ryt31/CS152-Project.git"
if not os.path.isdir("CS152-Project"):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO], check=True)
os.chdir("CS152-Project")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q "transformers>=4.45,<5" accelerate peft bitsandbytes

In [ ]:
# The run. Resumable: re-run this cell after a disconnect and it continues where it stopped.
# ~30-90 s per translation on a T4 -> 1-3 hours for 120. Keep the tab open.
!python pipeline/migrate.py --run-id {RUN_ID} --problems-dir benchmark --priority 1 \
    --backend hf --load-4bit --results-dir {RESULTS_DIR}

In [ ]:
# Progress: statuses and timing so far
import glob, json, statistics, collections
metas = [json.load(open(p)) for p in glob.glob(f"{RESULTS_DIR}/{RUN_ID}/*/*/meta.json")]
print(collections.Counter(m["status"] for m in metas))
print(collections.Counter(m.get("extraction") for m in metas if m["status"] == "ok"))
d = [m["duration_s"] for m in metas if m["status"] == "ok" and "duration_s" in m]
if d:
    print(f"translations: {len(d)}  median {statistics.median(d):.0f}s  total {sum(d)/3600:.1f} h")

## Scoring (local machine with Docker)

1. Download `MyDrive/CS152/results/raw/base01` from Drive (or zip it with the cell below) and place it at
   `results/raw/base01/` in your clone.
2. From the repo root: `python pipeline/collect.py --run-id base01`. This runs the harness on every pair
   folder and writes `results/base01.csv` plus a per-pair mean pass rate.

In [ ]:
!cd {RESULTS_DIR} && zip -qr /content/{RUN_ID}.zip {RUN_ID} && ls -la /content/{RUN_ID}.zip
from google.colab import files; files.download(f"/content/{RUN_ID}.zip")